In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import os
from PIL import Image
import numpy as np
import random

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Create OmniglotTrain Dataset Class
Implement OmniglotTrain class inheriting from torch.utils.data.Dataset with methods:
- __init__: Initialize with data path and transforms
- loadToMem: Load and rotate training images (0,90,180,270 degrees)
- __len__: Return dataset length
- __getitem__: Return image pairs with same/different class labels

In [3]:
class OmniglotTrain(Dataset):
    def __init__(self, dataPath, transform=None):
        super(OmniglotTrain, self).__init__()
        np.random.seed(0)
        self.transform = transform
        self.datas, self.num_classes = self.loadToMem(dataPath)

    def loadToMem(self, dataPath):
        print("begin loading training dataset to memory")
        datas = {}
        idx = 0
        for alphaPath in os.listdir(dataPath):
            for charPath in os.listdir(os.path.join(dataPath, alphaPath)):
                datas[idx] = []
                for samplePath in os.listdir(os.path.join(dataPath, alphaPath, charPath)):
                    filePath = os.path.join(dataPath, alphaPath, charPath, samplePath)
                    datas[idx].append(Image.open(filePath).convert('L'))
                idx += 1
        print("finish loading training dataset to memory")
        return datas, idx

    def __len__(self):
        # Dynamic calculation based on actual data loaded
        n_samples = self.num_classes * len(self.datas[0])
        # self.num_classes: actual number of character classes in small dataset
        # len(self.datas[0]): number of drawings per character (typically 20)
        return n_samples  # No need to multiply by 4 since rotations are handled by transforms

    def __getitem__(self, index):
        label = None
        image1 = None
        image2 = None
        if index % 2 == 1:
            label = 1.0
            idx1 = random.randint(0, self.num_classes - 1)
            image1 = random.choice(self.datas[idx1])
            image2 = random.choice(self.datas[idx1])
        else:
            label = 0.0
            idx1 = random.randint(0, self.num_classes - 1)
            idx2 = random.randint(0, self.num_classes - 1)
            while idx1 == idx2:
                idx2 = random.randint(0, self.num_classes - 1)
            image1 = random.choice(self.datas[idx1])
            image2 = random.choice(self.datas[idx2])

        if self.transform:
            image1 = self.transform(image1)
            image2 = self.transform(image2)
        return image1, image2, torch.from_numpy(np.array([label], dtype=np.float32))

# Create OmniglotTest Dataset Class
Implement OmniglotTest class inheriting from torch.utils.data.Dataset with methods:
- __init__: Initialize with data path, transforms, test times and way
- loadToMem: Load test images into memory
- __len__: Return dataset length
- __getitem__: Return image pairs for N-way testing

In [4]:
class OmniglotTest(Dataset):
    def __init__(self, dataPath, transform=None, times=200, way=20):
        np.random.seed(1)
        super(OmniglotTest, self).__init__()
        self.transform = transform
        self.times = times
        self.way = way
        self.img1 = None
        self.c1 = None
        self.datas, self.num_classes = self.loadToMem(dataPath)

    def loadToMem(self, dataPath):
        print("begin loading test dataset to memory")
        datas = {}
        idx = 0
        for alphaPath in os.listdir(dataPath):
            for charPath in os.listdir(os.path.join(dataPath, alphaPath)):
                datas[idx] = []
                for samplePath in os.listdir(os.path.join(dataPath, alphaPath, charPath)):
                    filePath = os.path.join(dataPath, alphaPath, charPath, samplePath)
                    datas[idx].append(Image.open(filePath).convert('L'))
                idx += 1
        print("finish loading test dataset to memory")
        return datas, idx

    def __len__(self):
        return self.times * self.way

    def __getitem__(self, index):
        idx = index % self.way
        label = None
        if idx == 0:
            self.c1 = random.randint(0, self.num_classes - 1)
            self.img1 = random.choice(self.datas[self.c1])
            img2 = random.choice(self.datas[self.c1])
            label = 1.0
        else:
            c2 = random.randint(0, self.num_classes - 1)
            while self.c1 == c2:
                c2 = random.randint(0, self.num_classes - 1)
            img2 = random.choice(self.datas[c2])
            label = 0.0

        if self.transform:
            img1 = self.transform(self.img1)
            img2 = self.transform(img2)
        return img1, img2, torch.from_numpy(np.array([label], dtype=np.float32))

In [5]:
import torchvision.transforms as transforms

# Define image transforms
train_transforms = transforms.Compose([
    transforms.RandomAffine(degrees = 15, #Radnom rotation +-15 degrees
                             translate=(3/105, 3/105)), # 3 pixel translation in any direction (image size is 105x105)
    transforms.ToTensor()
])

test_transforms = transforms.Compose([
    transforms.ToTensor()
])

# Create train and test dataset objects
train_dataset = OmniglotTrain(dataPath='/home/arnav/Siamese-one-shot/siamese-pytorch/omniglot/python/images_background_small1', transform=train_transforms)
test_dataset = OmniglotTest(dataPath='/home/arnav/Siamese-one-shot/siamese-pytorch/omniglot/python/images_evaluation', transform=test_transforms)



begin loading training dataset to memory
finish loading training dataset to memory
begin loading test dataset to memory
finish loading test dataset to memory


In [6]:
def create_validation_split(train_dataset, val_ratio=0.1):
    """
    Creates a validation dataset by splitting the training dataset
    val_ratio: percentage of training data to use for validation (0.1 = 10%)
    """
    # Calculate split sizes
    total_size = len(train_dataset)
    val_size = int(val_ratio * total_size)
    train_size = total_size - val_size
    
    # Split the dataset
    train_subset, val_subset = torch.utils.data.random_split(
        train_dataset, 
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)  # For reproducibility
    )
    
    return train_subset, val_subset


# Apply the split
train_subset, val_dataset = create_validation_split(train_dataset)

# Create data loaders
train_loader = DataLoader(train_subset, batch_size=128, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=4)

Here's an explanation of the Omniglot dataset structure:

### Dataset Overview
- Contains 1,623 different handwritten characters from 50 different alphabets
- Each character was drawn by 20 different people through Amazon Mechanical Turk
- Total images: 1,623 * 20 = 32,460 images

### Dataset Split
- **Background Set** (Training):
  - 30 alphabets 
  - Used for training/learning general character features
  
- **Evaluation Set** (Testing):
  - 20 alphabets
  - Used for one-shot learning evaluation
  - Completely separate alphabets from training set

### Directory Structure


omniglot/
├── images_background/    # Training set
│   ├── alphabet1/
│   │   ├── character1/
│   │   │   ├── 0001.png  # Drawing 1 
│   │   │   ├── 0002.png  # Drawing 2
│   │   │   └── ...       # 20 drawings total
│   │   └── character2/
│   └── alphabet2/
└── images_evaluation/    # Testing set
    └── [Similar structure]



### Key Properties
- Images are 105x105 pixels, black and white
- Characters are organized hierarchically by alphabet -> character -> drawing
- Training images are augmented with rotations (0°, 90°, 180°, 270°) 
- One-shot learning task uses within-alphabet discrimination (more challenging)
- Each image comes with stroke data (sequence of [x,y,t] coordinates)

### Dataset Philosophy
Designed to mimic human learning:
- Few examples per class (20)
- Large number of classes (1,623)
- Hierarchical structure (alphabets -> characters)
- Clean, simple binary images

This structure makes it ideal for one-shot learning research where the goal is to learn from very few examples.

In [7]:
#defining the architecture of our Siamese network:

import torch.nn as nn

class SiameseNetwork(nn.Module):
    def __init__(self):
        super(SiameseNetwork, self).__init__()
        #defining the convolutional layers: 
        self.conv = nn.Sequential(
            #layer1: 10*10 filters, 64 (4*16) output channels
            nn.Conv2d(1, 64, 10),  # 64@96*96
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 64@48*48

            #layer2: 7*7 filters, 128 (8*16) output channels
            nn.Conv2d(64, 128, 7),# 128@42*42
            nn.ReLU(),    
            nn.MaxPool2d(2,2),   # 128@21*21

            #layer3: 4*4 filters,  128(8*16) output channels
            nn.Conv2d(128, 128, 4),# 128@18*18
            nn.ReLU(), 
            nn.MaxPool2d(2,2), # 128@9*9

            # Layer 4: 4x4 filters, 256 (16*16) output channels
            nn.Conv2d(128, 256, 4),  # 256@6*6
            nn.ReLU(),  
        )

        self.fc = nn.Sequential(
            nn.Linear(256*6*6, 4096),
            nn.Sigmoid()
        )

        self.alpha = nn.Parameter(torch.ones(4096)) #learnable weights for component-wise distance

    def forward_one(self, x):
        x = self.conv(x)
        x = x.view(x.size()[0], -1)
        x = self.fc(x)
        return x
    
    def forward(self, input1, input2):
        output1 = self.forward_one(input1) #first embedding
        output2 = self.forward_one(input2) #second embedding

        distance = torch.abs(output1 - output2) * self.alpha #wieghted absolute differnece

        #final predictions
        similarity = torch.sigmoid(distance.sum(dim = 1, keepdim=True))
        return similarity




In [8]:
# # Define loss function and optimizer
# criterion = nn.BCELoss()
# model = SiameseNetwork().cuda() if torch.cuda.is_available() else SiameseNetwork()
# optimizer = torch.optim.Adam(model.parameters())

# # Training constants
# NUM_EPOCHS = 100
# TRAIN_BATCH_SIZE = 128
# TEST_BATCH_SIZE = 128

In [9]:
# def train_epoch(model, train_loader, criterion, optimizer):
#     model.train()
#     running_loss = 0.0
    
#     for batch_idx, (img1, img2, labels) in enumerate(train_loader):
#         # Move to GPU if available
#         img1, img2 = img1.cuda(), img2.cuda()
#         labels = labels.cuda()
        
#         # Zero gradients
#         optimizer.zero_grad()
        
#         # Forward pass
#         outputs = model(img1, img2)
#         loss = criterion(outputs, labels)
        
#         # Backward pass and optimize
#         loss.backward()
#         optimizer.step()
        
#         running_loss += loss.item()
        
#     return running_loss / len(train_loader)

In [10]:
# def evaluate(model, test_loader):
#     model.eval()
#     correct = 0
#     total = 0
    
#     with torch.no_grad():
#         for img1, img2, labels in test_loader:
#             img1, img2 = img1.cuda(), img2.cuda()
#             outputs = model(img1, img2)
#             predicted = (outputs > 0.5).float()
#             total += labels.size(0)
#             correct += (predicted.cpu() == labels).sum().item()
            
#     return correct / total

In [11]:
# # Training loop
# best_acc = 0.0
# for epoch in range(NUM_EPOCHS):
#     train_loss = train_epoch(model, train_loader, criterion, optimizer)
#     test_acc = evaluate(model, test_loader)
    
#     print(f'Epoch [{epoch+1}/{NUM_EPOCHS}]')
#     print(f'Train Loss: {train_loss:.4f}')
#     print(f'Test Accuracy: {test_acc:.4f}')
    
#     if test_acc > best_acc:
#         best_acc = test_acc
#         torch.save(model.state_dict(), 'best_model.pth')

In [12]:
import torch
import torch.nn as nn

def init_weights(m):
    if isinstance(m, nn.Conv2d): #for conv layers
        nn.init.normal_(m.weight, mean = 0.0, std = 0.01)
        nn.init.normal_(m.bias, mean = 0.5, std=0.01)
    elif isinstance(m, nn.Linear): #for fc layer
        nn.init.normal_(m.weight, mean = 0, std = 0.2)
        nn.init.normal_(m.bias, mean =0.5, std=0.01)


In [13]:
class customOptimizer:
    def __init__(self, params, initial_lr=0.01, initial_momentum = 0.5, final_momentum=0, l2_reg = 0.00001):
        self.params = list(params)
        self.lrs = [initial_lr for _ in self.params]
        self.momentums = [initial_momentum for _ in self.params]
        self.final_momentums = [final_momentum for _ in self.params]
        self.l2_reg = l2_reg
        self.velocities = [torch.zeros_like(p.data) for p in self.params]
        self.epoch = 0

    def step(self):
        for i, p in enumerate(self.params):
            if p.grad is None:
                continue

            #L2 regularization:
            p.grad.data.add_(self.l2_reg * p.data) #formula is gradient += l2_reg * parameter

            #update velocity
            self.velocities[i] = self.momentums[i] * self.velocities[i] + self.lrs[i]  * p.grad.data

            #update params 
            p.data.add_(-self.velocities[i])
    
    def update_schedule(self, epoch):
        self.epoch = epoch

        # 1% learning rate decay
        for  i in range(len(self.lrs)):
            self.lrs[i] *= 0.99

        # momentum increase
        progress = min(1.0, epoch / 200.0) # for a total of 200 epochs
        for i in range(len(self.momentums)):
            self.momentums[i] = self.final_momentums[i] *progress + 0.5*(1-progress)
    
    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.data.zero_()
            
    def state_dict(self):
        return {
            'lrs': self.lrs,
            'momentums': self.momentums,
            'velocities': self.velocities,
            'epoch': self.epoch
        }
    
    def load_state_dict(self, state_dict):
        self.lrs = state_dict['lrs']
        self.momentums = state_dict['momentums']
        self.velocities = state_dict['velocities']
        self.epoch = state_dict['epoch']
                

In [14]:
#training setuo

model = SiameseNetwork().to(device)
model.apply(init_weights)
optimizer = customOptimizer(model.parameters(), initial_lr=0.01, initial_momentum = 0.5, final_momentum=0.9, l2_reg = 0.00001)
criterion = nn.BCELoss()

In [15]:
# Early stopping setup
best_val_acc = 0
patience = 20
patience_counter = 0



In [16]:
import torch
import torch.nn as nn
import time
import os
from tqdm import tqdm
import numpy as np

def create_validation_tasks(val_dataset, num_tasks=320):
    """Create one-shot validation tasks"""
    val_tasks = []
    for _ in range(num_tasks):
        # Random pair from validation set
        img1, img2, label = val_dataset[np.random.randint(len(val_dataset))]
        val_tasks.append((img1, img2, label))
    return val_tasks

def train_siamese_network(
    model,
    train_loader,
    val_dataset,
    optimizer,
    criterion,
    num_epochs=200,
    device='cuda',
    checkpoint_dir='checkpoints'
):
    # Create checkpoint directory
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    # Setup tracking variables
    best_val_acc = 0.0
    patience = 20
    patience_counter = 0
    train_losses = []
    val_accuracies = []
    
    # Create validation tasks
    val_tasks = create_validation_tasks(val_dataset)
    
    # Training loop
    for epoch in range(num_epochs):
        # Update learning schedule
        optimizer.update_schedule(epoch)
        
        # Training phase
        model.train()
        epoch_loss = 0.0
        start_time = time.time()
        
        # Progress bar for training
        with tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}') as pbar:
            for batch_idx, (img1, img2, labels) in enumerate(pbar):
                # Move to device
                img1, img2 = img1.to(device), img2.to(device)
                labels = labels.to(device)
                
                # Zero gradients
                optimizer.zero_grad()
                
                # Forward pass
                outputs = model(img1, img2)
                loss = criterion(outputs, labels)
                
                # Backward pass and optimize
                loss.backward()
                optimizer.step()
                
                # Update metrics
                epoch_loss += loss.item()
                pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_train_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # Validation phase
        model.eval()
        correct = 0
        total = len(val_tasks)
        
        with torch.no_grad():
            for img1, img2, label in val_tasks:
                img1, img2 = img1.to(device).unsqueeze(0), img2.to(device).unsqueeze(0)
                output = model(img1, img2)
                predicted = (output > 0.5).float()
                correct += (predicted.cpu().item() == label.item())
        
        val_acc = correct / total
        val_accuracies.append(val_acc)
        
        # Print epoch summary
        epoch_time = time.time() - start_time
        print(f'\nEpoch {epoch+1}/{num_epochs}:')
        print(f'Training Loss: {avg_train_loss:.4f}')
        print(f'Validation Accuracy: {val_acc:.4f}')
        print(f'Time: {epoch_time:.2f}s')
        
        # Save best model and check early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
            }, os.path.join(checkpoint_dir, 'best_model.pth'))
            patience_counter = 0
        else:
            patience_counter += 1
            
        # Early stopping check
        if patience_counter >= patience:
            print(f'Early stopping triggered after epoch {epoch+1}')
            break
            
        # Save checkpoint every 10 epochs
        if (epoch + 1) % 10 == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_losses': train_losses,
                'val_accuracies': val_accuracies,
            }, os.path.join(checkpoint_dir, f'checkpoint_epoch_{epoch+1}.pth'))
    
    return train_losses, val_accuracies

# Usage
if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = SiameseNetwork().to(device)
    model.apply(init_weights)
    
    optimizer = customOptimizer(
        model.parameters(),
        initial_lr=0.01,
        initial_momentum=0.5,
        final_momentum=0.9,
        l2_reg=0.00001
    )
    
    criterion = nn.BCELoss()
    
    train_losses, val_accuracies = train_siamese_network(
        model=model,
        train_loader=train_loader,
        val_dataset=val_dataset,
        optimizer=optimizer,
        criterion=criterion,
        device=device
    )

Epoch 1/200: 100%|██████████| 20/20 [00:05<00:00,  3.88it/s, loss=43.7500]



Epoch 1/200:
Training Loss: 41.5697
Validation Accuracy: 0.4469
Time: 5.95s


Epoch 2/200: 100%|██████████| 20/20 [00:04<00:00,  4.04it/s, loss=68.7500]



Epoch 2/200:
Training Loss: 50.5859
Validation Accuracy: 0.4469
Time: 5.72s


Epoch 3/200: 100%|██████████| 20/20 [00:04<00:00,  4.04it/s, loss=37.5000]



Epoch 3/200:
Training Loss: 49.2188
Validation Accuracy: 0.4469
Time: 5.72s


Epoch 4/200:  50%|█████     | 10/20 [00:02<00:02,  3.48it/s, loss=46.0938]


KeyboardInterrupt: 